---
## World + LLM setup


In [1]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()
symbol_type = Body


`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


In [2]:
from pycram.datastructures.enums import Arms
from pycram.exceptions import MotionDidNotFinish
from pycram.locations.locations import CostmapLocation
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction
from semantic_digital_twin.datastructures.definitions import TorsoState


def prepare_robot_for_pickup(context):
    setup_plan = sequential(
        [
            ParkArmsAction(Arms.BOTH),
            MoveTorsoAction(TorsoState.HIGH),
        ],
        context=context,
    )
    with simulated_robot:
        setup_plan.perform()


def build_navigate_then_pick(action_instance, context):
    pickup_pose = CostmapLocation(
        target=action_instance.object_designator.global_pose,
        reachable=True,
        reachable_arm=action_instance.arm,
        grasp_description=action_instance.grasp_description,
        context=context,
    ).ground()

    return sequential(
        [
            NavigateAction(pickup_pose, keep_joint_states=True),
            PickUpAction(
                object_designator=action_instance.object_designator,
                arm=action_instance.arm,
                grasp_description=action_instance.grasp_description,
            ),
        ],
        context=context,
    )


def print_pickup_action(action_instance):
    print("Action type:", type(action_instance).__name__)
    print("Object     :", action_instance.object_designator)
    print("Arm        :", action_instance.arm)
    print("Grasp      :", action_instance.grasp_description)
    print("Offset     :", action_instance.grasp_description.manipulation_offset)


In [3]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()


In [4]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")


ROS2 publishers started


In [5]:
from dotenv import load_dotenv
load_dotenv("../.env")

from llmr.reasoning.llm_provider import make_llm, LLMProvider

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print("LLM ready:", getattr(llm, "model_name", llm))


LLM ready: gpt-4o-mini


In [6]:
# Shared instruction and match factory — call fresh_match() for each section
# to get an unsolved Match expression every time.

from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.datastructures.grasp import GraspDescription
from llmr.backend import LLMBackend

INSTRUCTION = "pick up the milk from the table"

def fresh_match():
    return Match(PickUpAction)(
        object_designator=...,
        arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            manipulator=...,
        ),
    )

print("Match factory ready.")


Match factory ready.


---
## Section 1 — Base backend (slot filling only)

`ActionAnnotationBundle.slot_filling` contains the raw LLM output:
- `action_type` — echoed action class name
- `slots` — one `SlotValue` per free slot, with `field_name`, `value`, `entity_description`
- `overall_reasoning` — LLM explanation


In [ ]:
backend_base = LLMBackend(
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    strict_required=False,   # don't raise on unresolved — we want to see partial results
)

action_base = next(iter(backend_base.evaluate(fresh_match())))
print("Resolved action type:", type(action_base).__name__)


In [ ]:
# type(action_base)


In [ ]:
# Full ActionAnnotationBundle dump — inspect every field
import json

sem = backend_base.semantics
print(json.dumps(sem.model_dump(), indent=2, default=str))


In [ ]:
# Inspect slot_filling in detail
sf = sem.slot_filling
print(f"action_type       : {sf.action_type}")
print(f"overall_reasoning : {sf.overall_reasoning}")
print(f"\nSlots ({len(sf.slots)}):")
for slot in sf.slots:
    print(f"  field_name        : {slot.field_name}")
    print(f"  value             : {slot.value}")
    if slot.entity_description:
        ed = slot.entity_description
        print(f"  entity_description: name={ed.name!r}, type={ed.semantic_type!r},"
              f" spatial={ed.spatial_context!r}")
    print(f"  reasoning         : {slot.reasoning}")
    print()


In [ ]:
# The resolved action instance itself
print("arm  :", action_base.arm)
# print("grasp:", action_base.grasp_description)
print("obj  :", action_base.object_designator)


---
## Section 2 — With FrameNetReasoner

`FrameNetReasoner` runs one LLM call after slot filling and populates
`semantics.frames` with a `FrameNetAnnotation`:

- `frame` — official FrameNet frame name (e.g. `Getting`)
- `framenet` — snake_case action label
- `lexical_unit` — lemma.pos (e.g. `pick_up.v`)
- `core` — THEME, AGENT, SOURCE, GOAL, etc.
- `peripheral` — LOCATION, MANNER, DIRECTION, etc.


In [ ]:
from llmr.reasoning.framenet_reasoner import FrameNetReasoner

backend_fn = LLMBackend(
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    reasoners=[FrameNetReasoner(llm=llm)],
)

action_fn = next(iter(backend_fn.evaluate(fresh_match())))
print("Resolved:", type(action_fn).__name__)


In [ ]:
# FrameNetAnnotation — full dump
import json
print(json.dumps(backend_fn.semantics.frames.model_dump(by_alias=True), indent=2, default=str))


In [ ]:
# backend_fn.semantics.model_dump()


In [ ]:
# Read individual FrameNet fields
fn = backend_fn.semantics.frames
print(f"Frame        : {fn.frame}")
print(f"Framenet     : {fn.framenet}")
print(f"Lexical unit : {fn.lexical_unit}")

print("\nCore elements:")
for k, v in fn.core.model_dump().items():
    if v is not None:
        print(f"  {k:<25}: {v}")

print("\nPeripheral elements:")
for k, v in fn.peripheral.model_dump().items():
    if v is not None:
        print(f"  {k:<25}: {v}")


---
## Section 3 — With FlanaganReasoner

`FlanaganReasoner` uses a **2-call LLM pipeline** and populates
`semantics.motion_phases` with a `FlanaganMotionPlan`:

- **Call 1** — free-form phase decomposition: LLM reasons freely about which
  motion phases are needed (no hardcoded vocabulary restriction).
- **Call 2** — holistic annotation: all phases annotated in one pass, grounded
  with the serialized world scene (`world_context`) and resolved action parameters
  (`match_data` resolved slots — arm, grounded object, grasp type).

Each `MotionPhase` in `semantics.motion_phases.phases` contains:
- `phase`, `target_object`, `description`, `symbol`
- `preconditions`, `goal_state`
- `force_dynamics`, `sensory_feedback`
- `failure_and_recovery`, `temporal_constraints`

If you later want to query these phases through the hypothesis layer instead of reading the raw sidecar directly, use `FlanaganFamily.make_manager(graph)` during evaluation and `FlanaganFamily.make_view(graph)` for typed graph queries.


In [7]:
from llmr.reasoning.flanagan_reasoner import FlanaganReasoner

backend_fl = LLMBackend(
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    reasoners=[FlanaganReasoner(llm=llm)],
)

action_fl = next(iter(backend_fl.evaluate(fresh_match())))
print("Resolved:", type(action_fl).__name__)


Resolved: PickUpAction


dict_keys(['action_type', 'instruction', 'classification', 'slot_filling', 'motion_phases', 'frames', 'preconditions', 'postconditions', 'affordances', 'extra'])

In [8]:
print(type(action_fl))
print(action_fl.object_designator)
print(action_fl.arm)
print(action_fl.grasp_description.approach_direction)
print(action_fl.grasp_description.vertical_alignment)
print(action_fl.grasp_description.manipulator.name)


<class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>
Body(name=PrefixedName('None/milk.stl'), id=UUID('28756eca-bb09-496d-9a9a-ff36c66ec729'), index=219)
RIGHT
ApproachDirection.FRONT
VerticalAlignment.TOP
pr2/right_gripper


In [9]:
# FlanaganMotionPlan — full dump (can be large)
import json
print(json.dumps(backend_fl.semantics.motion_phases.model_dump(), indent=2, default=str))


{
  "instruction": "pick up the milk from the table",
  "phases": [
    {
      "phase": "Approach",
      "target_object": "milk",
      "description": "move to milk on table",
      "symbol": "->[ robot approachs milk]",
      "goal_state": {
        "arm_near_milk": true,
        "distance_to_milk_m": 0.2
      },
      "preconditions": {
        "milk_visible": true,
        "path_to_milk_clear": true,
        "arm_in_starting_position": true
      },
      "force_dynamics": {},
      "sensory_feedback": {
        "camera_milk_detected": true,
        "RIGHT_arm_joint_position": "approaching"
      },
      "failure_and_recovery": {
        "possible_failures": [
          "milk not visible",
          "path blocked by obstacle"
        ],
        "recovery_strategies": [
          "replan arm trajectory",
          "request operator assistance"
        ]
      },
      "temporal_constraints": {
        "max_duration_sec": 3.0,
        "urgency": "medium"
      }
    },
    {
     

In [ ]:
# # Per-phase summary — one line per phase, then detail on demand
# fp = backend_fl.semantics.motion_phases
# print(f"Instruction : {fp.instruction}")
# print(f"Phase count : {len(fp.phases)}\n")
#
# for i, phase in enumerate(fp.phases, 1):
#     print(f"--- Phase {i}: [{phase.phase}] → {phase.target_object} ---")
#     print(f"  symbol      : {phase.symbol}")
#     if phase.description:
#         print(f"  description : {phase.description}")
#     if phase.preconditions:
#         print(f"  preconditions : {phase.preconditions}")
#     if phase.goal_state:
#         print(f"  goal_state    : {phase.goal_state}")
#     if phase.force_dynamics:
#         print(f"  force_dynamics: {phase.force_dynamics}")
#     if phase.temporal_constraints:
#         print(f"  timing        : {phase.temporal_constraints}")
#     print()


In [21]:
prepare_robot_for_pickup(context)

role1_plan_node = build_navigate_then_pick(action_fl, context)
print("Role 1 Navigate + PickUp plan ready:", role1_plan_node)
with simulated_robot:
    try:
        role1_plan_node.perform()
    except MotionDidNotFinish:
        print("Role 1 motion did not finish; pickup parameters and plan construction were still exercised.")


INFO:base.py::60 perform Performing action ParkArmsAction
INFO:base.py::60 perform Performing action MoveTorsoAction
INFO:base.py::60 perform Performing action NavigateAction


Role 1 Navigate + PickUp plan ready: SequentialNode(plan=Plan with 3 nodes, status=<TaskStatus.CREATED: 0>, start_time=datetime.datetime(2026, 4, 22, 12, 46, 5, 929182), end_time=None, reason=None, result=None)


INFO:base.py::60 perform Performing action PickUpAction
INFO:base.py::60 perform Performing action ReachAction


---
## Section 4 — Full composite (both reasoners)

All three `ActionAnnotationBundle` fields populated at once:
`slot_filling` + `frames` + `motion_phases`.


In [ ]:
backend_full = LLMBackend(
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    reasoners=[
        FrameNetReasoner(llm=llm),
        FlanaganReasoner(llm=llm),
    ],
)

action_full = next(iter(backend_full.evaluate(fresh_match())))
print("Resolved:", type(action_full).__name__)


In [ ]:
# Top-level overview of what is populated
sem_full = backend_full.semantics

print("ActionAnnotationBundle overview")
print("========================")
print(f"action_type   : {sem_full.action_type}")
print(f"instruction   : {sem_full.instruction!r}")
print(f"classification: {sem_full.classification}")
print(f"slot_filling  : {type(sem_full.slot_filling).__name__} — {len(sem_full.slot_filling.slots)} slots")
print(f"frames        : {type(sem_full.frames).__name__} — frame={sem_full.frames.frame!r}")
print(f"motion_phases : {type(sem_full.motion_phases).__name__} — {len(sem_full.motion_phases.phases)} phases")
print(f"preconditions : {sem_full.preconditions}")
print(f"postconditions: {sem_full.postconditions}")
print(f"affordances   : {sem_full.affordances}")
print(f"extra         : {sem_full.extra}")


In [ ]:
# Full JSON tree — all nested fields
print(json.dumps(sem_full.model_dump(by_alias=True), indent=2, default=str))


In [ ]:
# Side-by-side: FrameNet theme/source vs slot-filler object designator
fn_core = sem_full.frames.core
sf_slots = {s.field_name: s for s in sem_full.slot_filling.slots}

print("FrameNet THEME  :", fn_core.theme)
print("FrameNet SOURCE :", fn_core.source)
print("FrameNet AGENT  :", fn_core.agent)
print()
obj_slot = sf_slots.get("object_designator")
if obj_slot and obj_slot.entity_description:
    print("Slot-filler object designator:")
    print("  name          :", obj_slot.entity_description.name)
    print("  semantic_type :", obj_slot.entity_description.semantic_type)
    print("  spatial_context:", obj_slot.entity_description.spatial_context)


In [ ]:
# Quick Flanagan phase name list from the full composite run
phases_summary = [
    f"{p.phase} → {p.target_object}" for p in sem_full.motion_phases.phases
]
for i, s in enumerate(phases_summary, 1):
    print(f"  {i}. {s}")


---
## Section 5 — Serialise and reload `ActionAnnotationBundle`

The full sidecar can be serialised to JSON (e.g. for logging, replay, or offline
analysis) and reconstructed losslessly.


In [ ]:
from llmr.schemas import ActionAnnotationBundle

# Serialise
json_str = sem_full.model_dump_json(by_alias=True)
print("Serialised length (bytes):", len(json_str))

# Reconstruct
reloaded = ActionAnnotationBundle.model_validate_json(json_str)
print("Reloaded action_type  :", reloaded.action_type)
print("Reloaded frame        :", reloaded.frames.frame)
print("Reloaded phase count  :", len(reloaded.motion_phases.phases))
print("Reloaded slot count   :", len(reloaded.slot_filling.slots))


In [12]:
backend_fl.semantics.model_dump().keys()


dict_keys(['action_type', 'instruction', 'classification', 'slot_filling', 'motion_phases', 'frames', 'preconditions', 'postconditions', 'affordances', 'extra'])

In [20]:
backend_fl.semantics.motion_phases.model_dump()


{'instruction': 'pick up the milk from the table',
 'phases': [{'phase': 'Approach',
   'target_object': 'milk',
   'description': 'move to milk on table',
   'symbol': '->[ robot approachs milk]',
   'goal_state': {'arm_near_milk': True, 'distance_to_milk_m': 0.2},
   'preconditions': {'milk_visible': True,
    'path_to_milk_clear': True,
    'arm_in_starting_position': True},
   'force_dynamics': {},
   'sensory_feedback': {'camera_milk_detected': True,
    'RIGHT_arm_joint_position': 'approaching'},
   'failure_and_recovery': {'possible_failures': ['milk not visible',
     'path blocked by obstacle'],
    'recovery_strategies': ['replan arm trajectory',
     'request operator assistance']},
   'temporal_constraints': {'max_duration_sec': 3.0, 'urgency': 'medium'}},
  {'phase': 'Grasp',
   'target_object': 'milk',
   'description': 'grip the milk container',
   'symbol': '->[ robot grasps milk]',
   'goal_state': {'milk_in_RIGHT_gripper': True, 'gripper_closed': True},
   'preconditi